In [ ]:
# Contributors: [Group 5 — fill in names]
# Course: Recommendation Engines, IE University 2025-26
# Notebook 05: Comparative Evaluation (Phase 3)
#
# This notebook loads pre-computed results from notebooks 01-04 (pkl files)
# and does NOT retrain any models. Run notebooks 01-04 first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils import *

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

np.random.seed(42)

## 1. Load Results from Notebooks 01–04

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

data_np  = load_pkl('../results/01_non_personalized.pkl')
data_cf  = load_pkl('../results/02_collaborative_filtering.pkl')
data_cb  = load_pkl('../results/03_content_based.pkl')
data_ctx = load_pkl('../results/04_context_aware.pkl')

print('Loaded all result files.')
print('NP  keys:', list(data_np.keys()))
print('CF  keys:', list(data_cf.keys()))
print('CB  keys:', list(data_cb.keys()))
print('CTX keys:', list(data_ctx.keys()))

In [ ]:
# Retrieve pre-computed metric dicts
results_np  = data_np['results_np']      # keys: 'Global Mean', 'Item Mean', 'Damped Mean'
results_cf  = data_cf['results_cf']      # keys: 'SVD'
results_cb  = data_cb['results_cb']
results_ctx = data_ctx['results_ctx']

# Item feature matrix and mappings (needed for diversity / serendipity)
item_features = data_cb['item_features']
item_to_idx   = data_cb['item_to_idx']

# Recommendation lists per model
recs_np  = data_np['recs_np']    # {model_name: {user_id: [item_ids]}}
recs_cf  = data_cf['recs_cf']    # {'SVD': {user_id: [item_ids]}}
recs_cb  = data_cb['recs_cb']    # {user_id: [item_ids]}
recs_ctx = data_ctx['recs_ctx']  # {user_id: [item_ids]}

print(f'item_features shape: {item_features.shape}')

## 2. Reload Data for Lookup Structures

We need `train_user_items` and `test_user_relevant` to compute serendipity.

In [ ]:
df, _ = load_data()
train_df, test_df, _ = temporal_train_test_split(df)
lookups = build_lookup_structures(train_df, test_df)
train_user_items   = lookups['train_user_items']
test_user_relevant = lookups['test_user_relevant']

## 3. Compute Diversity & Serendipity for All Models

Notebooks 01 and 02 saved recommendation lists but could not compute diversity / serendipity
because `item_features` is only built in notebook 03. We compute all beyond-accuracy metrics
here using the saved recommendation lists.

In [ ]:
def compute_beyond_accuracy(rec_dict, label):
    rec_lists  = list(rec_dict.values())
    div  = diversity_intra_list(rec_lists, item_features, item_to_idx)
    ser  = serendipity(rec_dict, train_user_items, test_user_relevant, item_features, item_to_idx)
    print(f'  {label}: Diversity={div:.4f}  Serendipity={ser:.4f}')
    return div, ser

print('Computing beyond-accuracy metrics...')
div_gm,  ser_gm  = compute_beyond_accuracy(recs_np['Global Mean'],  'Global Mean')
div_im,  ser_im  = compute_beyond_accuracy(recs_np['Item Mean'],    'Item Mean')
div_dm,  ser_dm  = compute_beyond_accuracy(recs_np['Damped Mean'],  'Damped Mean')
div_svd, ser_svd = compute_beyond_accuracy(recs_cf['SVD'],          'SVD (CF)')
div_cb,  ser_cb  = compute_beyond_accuracy(recs_cb,                 'Content-Based')
div_ctx, ser_ctx = compute_beyond_accuracy(recs_ctx,                'Context-Aware')

## 4. Standardised Comparison Table

The table below follows the assignment rubric structure.

In [ ]:
def _r(results, model_key, metric):
    return results.get(model_key, {}).get(metric)

rows = [
    {
        'Approach':      'Non-Personalized (Damped Mean)',
        'RMSE':          _r(results_np, 'Damped Mean', 'RMSE'),
        'MAE':           _r(results_np, 'Damped Mean', 'MAE'),
        'Precision@K':   _r(results_np, 'Damped Mean', f'Precision@{K}'),
        'Recall@K':      _r(results_np, 'Damped Mean', f'Recall@{K}'),
        'NDCG@K':        _r(results_np, 'Damped Mean', f'NDCG@{K}'),
        'Coverage':      _r(results_np, 'Damped Mean', 'Coverage'),
        'Diversity':     div_dm,
        'Serendipity':   ser_dm,
        'Context':       'None',
    },
    {
        'Approach':      'Collaborative Filtering (SVD)',
        'RMSE':          _r(results_cf, 'SVD', 'RMSE'),
        'MAE':           _r(results_cf, 'SVD', 'MAE'),
        'Precision@K':   _r(results_cf, 'SVD', f'Precision@{K}'),
        'Recall@K':      _r(results_cf, 'SVD', f'Recall@{K}'),
        'NDCG@K':        _r(results_cf, 'SVD', f'NDCG@{K}'),
        'Coverage':      _r(results_cf, 'SVD', 'Coverage'),
        'Diversity':     div_svd,
        'Serendipity':   ser_svd,
        'Context':       'None',
    },
    {
        'Approach':      'Content-Based (TF-IDF + Features)',
        'RMSE':          results_cb.get('RMSE'),
        'MAE':           results_cb.get('MAE'),
        'Precision@K':   results_cb.get(f'Precision@{K}'),
        'Recall@K':      results_cb.get(f'Recall@{K}'),
        'NDCG@K':        results_cb.get(f'NDCG@{K}'),
        'Coverage':      results_cb.get('Coverage'),
        'Diversity':     div_cb,
        'Serendipity':   ser_cb,
        'Context':       'None',
    },
    {
        'Approach':      'Context-Aware (SVD + Temporal Bias)',
        'RMSE':          results_ctx.get('RMSE'),
        'MAE':           results_ctx.get('MAE'),
        'Precision@K':   results_ctx.get(f'Precision@{K}'),
        'Recall@K':      results_ctx.get(f'Recall@{K}'),
        'NDCG@K':        results_ctx.get(f'NDCG@{K}'),
        'Coverage':      results_ctx.get('Coverage'),
        'Diversity':     div_ctx,
        'Serendipity':   ser_ctx,
        'Context':       'month × day-of-week × time-period',
    },
]

comparison_df = pd.DataFrame(rows).set_index('Approach')
float_cols = ['RMSE', 'MAE', 'Precision@K', 'Recall@K', 'NDCG@K', 'Coverage', 'Diversity', 'Serendipity']
comparison_df[float_cols] = comparison_df[float_cols].apply(pd.to_numeric, errors='coerce')

print(f'\n=== Comparative Evaluation (K={K}) ===')
print(comparison_df.to_string(float_format='{:.4f}'.format))

## 5. Metric Analysis

### 5.1 Prediction Accuracy (RMSE, MAE)

In [ ]:
best_rmse = comparison_df['RMSE'].idxmin()
best_mae  = comparison_df['MAE'].idxmin()
print('=== Prediction Accuracy Analysis ===')
print(f'Best RMSE: {best_rmse} ({comparison_df.loc[best_rmse, "RMSE"]:.4f})')
print(f'Best MAE:  {best_mae}  ({comparison_df.loc[best_mae, "MAE"]:.4f})')
print()
print('Interpretation:')
print('  - SVD and Context-Aware outperform Non-Personalized baselines by capturing')
print('    latent user-item interactions absent in global/item-level averages.')
print('  - Content-Based has higher RMSE because cosine similarity in TF-IDF space')
print('    does not perfectly proxy rating similarity — items can be textually similar')
print('    yet rated very differently (e.g. different price tiers in the same category).')
print('  - Context-Aware offers a marginal MAE improvement over plain SVD by correcting')
print('    for systematic temporal biases (holiday effect, recency trend).')

### 5.2 Ranking Quality (Precision@K, Recall@K, NDCG@K)

In [ ]:
best_prec = comparison_df['Precision@K'].idxmax()
best_ndcg = comparison_df['NDCG@K'].idxmax()
print('=== Ranking Quality Analysis ===')
print(f'Best Precision@{K}: {best_prec} ({comparison_df.loc[best_prec, "Precision@K"]:.4f})')
print(f'Best NDCG@{K}:      {best_ndcg} ({comparison_df.loc[best_ndcg, "NDCG@K"]:.4f})')
print()
print('Why are absolute values low?')
print(f'  - The item catalog has {len(item_to_idx):,} items. Even a perfect ranker')
print(f'    achieves at most K/{len(item_to_idx):,} precision if users interact with only a few items.')
print('  - The test window is 6 months; most users have <5 relevant test items.')
print('  - The 2000-candidate sampling in SVD/Context-Aware biases toward items the user')
print('    has not seen — some relevant items may not be sampled.')
print()
print('  Non-Personalized Damped Mean achieves the highest Precision@K and NDCG@K')
print('  because it consistently recommends universally popular (high-rated) items that')
print('  many users happen to find relevant. Personalised models sacrifice this broad')
print('  coverage for individual specificity, which requires denser feedback to shine.')

### 5.3 Beyond-Accuracy Metrics (Coverage, Diversity, Serendipity)

In [ ]:
best_cov  = comparison_df['Coverage'].idxmax()
best_div  = comparison_df['Diversity'].idxmax()
best_ser  = comparison_df['Serendipity'].idxmax()
print('=== Beyond-Accuracy Analysis ===')
print(f'Best Coverage:    {best_cov}  ({comparison_df.loc[best_cov,  "Coverage"]:.4f})')
print(f'Best Diversity:   {best_div}  ({comparison_df.loc[best_div,  "Diversity"]:.4f})')
print(f'Best Serendipity: {best_ser}  ({comparison_df.loc[best_ser,  "Serendipity"]:.4f})')
print()
print('Trade-offs:')
print('  - Non-Personalized has near-zero coverage: all users get the same top-10,')
print('    so only ~10 unique items surface across 1000 users.')
print('  - Content-Based achieves higher coverage and diversity by routing different users')
print('    to different niche items via their personal profiles.')
print('  - SVD has moderate coverage from the random 2000-candidate sampling.')
print('  - Accuracy and diversity are in tension: optimising one often harms the other.')
print('    A hybrid model (e.g. combining CF for accuracy with CB for discovery) could')
print('    balance both objectives simultaneously.')

## 6. Visualizations

In [ ]:
# Grouped bar chart
plot_metrics = ['RMSE', 'MAE', 'Precision@K', 'Recall@K', 'NDCG@K', 'Coverage', 'Diversity', 'Serendipity']
models       = comparison_df.index.tolist()
n_metrics    = len(plot_metrics)
n_models     = len(models)
x            = np.arange(n_metrics)
width        = 0.18
colors       = ['#4878cf', '#6acc65', '#d65f5f', '#b47cc7']

fig, ax = plt.subplots(figsize=(15, 6))
for i, (model, color) in enumerate(zip(models, colors)):
    vals = [comparison_df.loc[model, m] for m in plot_metrics]
    ax.bar(x + i * width, vals, width, label=model.split(' (')[0], color=color, edgecolor='white')

ax.set_title('Model Comparison Across All Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels(plot_metrics, rotation=20, ha='right')
ax.legend(fontsize=9)
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
# Radar / spider chart
radar_metrics = ['Precision@K', 'Recall@K', 'NDCG@K', 'Coverage', 'Diversity', 'Serendipity']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model, color in zip(models, colors):
    vals = []
    for m in radar_metrics:
        col_max = comparison_df[m].max()
        v = comparison_df.loc[model, m]
        vals.append(float(v / col_max) if col_max and not np.isnan(v) else 0.0)
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=model.split(' (')[0], color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics)
ax.set_ylim(0, 1)
ax.set_title('Model Strengths (normalised per metric)', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Metrics heatmap
heatmap_data = comparison_df[float_cols].copy()

# Normalise each column to [0,1] for visual clarity
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())
# For RMSE and MAE lower is better — invert
heatmap_norm[['RMSE', 'MAE']] = 1 - heatmap_norm[['RMSE', 'MAE']]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_norm.T,
    annot=heatmap_data.T.round(4),
    fmt='g',
    cmap='RdYlGn',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Normalised score (higher = better)'}
)
ax.set_title('Metrics Heatmap (raw values annotated, colour = normalised rank)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 7. Train/Test Split Methodology

We use a **temporal 80/20 split** rather than a random split for three reasons:

1. **No data leakage** — a random split could place an early 2014 interaction in the training
   set while a 2013 interaction from the same user ends up in test, letting the model "cheat"
   by training on future information about the user.

2. **Realistic deployment simulation** — in production, models are trained on historical data
   and evaluated on future interactions. The temporal split mirrors this: train on 1999–Jan 2014,
   evaluate on Jan–Jul 2014.

3. **Cold-start stress test** — the temporal boundary exposes 12,726 users and 10,736 items
   that appear only in the test window, naturally evaluating cold-start robustness without
   artificially withholding data.

## 8. A/B Test Design

### Objective
Determine whether the Context-Aware model (treatment) produces a measurable improvement in
user engagement and satisfaction over the Damped Mean baseline (control) in a live system.

### Hypotheses
- **H₀**: CTR(treatment) = CTR(control) — context-awareness has no effect on click-through rate.
- **H₁**: CTR(treatment) > CTR(control) — the context-aware model achieves higher CTR.

### Design

| Parameter | Value |
|---|---|
| Control | Damped Mean (non-personalized baseline) |
| Treatment | Context-Aware SVD |
| Traffic split | 50% / 50% random user assignment |
| Duration | 2 weeks (14 days) |
| Primary metric | Click-through rate (CTR) on recommendations |
| Secondary metrics | Average rating of clicked items, conversion rate |
| Significance level | α = 0.05 (two-tailed) |
| Statistical test | Two-proportion z-test |

### Sample Size Estimation

In [ ]:
from scipy import stats

# Assumptions
p_control   = 0.05   # baseline CTR: 5%
min_lift    = 0.01   # minimum detectable effect: +1 pp (relative ~20%)
p_treatment = p_control + min_lift
alpha       = 0.05
power       = 0.80

# Two-proportion z-test sample size formula
z_alpha = stats.norm.ppf(1 - alpha / 2)  # two-tailed
z_beta  = stats.norm.ppf(power)
p_bar   = (p_control + p_treatment) / 2
n_per_group = (
    (z_alpha * (2 * p_bar * (1 - p_bar)) ** 0.5
     + z_beta * (p_control * (1 - p_control) + p_treatment * (1 - p_treatment)) ** 0.5) ** 2
    / (p_treatment - p_control) ** 2
)

print('=== A/B Test Sample Size Estimation ===')
print(f'Baseline CTR (control):     {p_control:.0%}')
print(f'Target CTR (treatment):     {p_treatment:.0%}  (+{min_lift:.0%} absolute)')
print(f'Alpha: {alpha}  |  Power: {power}')
print(f'Required users per group:   {int(np.ceil(n_per_group)):,}')
print(f'Total users needed:         {int(np.ceil(n_per_group)) * 2:,}')
print()
print('With ~129K test users available, a 2-week experiment at 50/50 split')
print('comfortably exceeds the required sample size.')

### Guardrail Metrics

Before declaring a winner, we also monitor:

| Guardrail | Threshold | Rationale |
|---|---|---|
| P99 latency | < 200 ms | Context-Aware runs SVD prediction at request time |
| Coverage drop | < 5% relative | Ensure the model still surfaces diverse items |
| Revenue per session | ≥ control | Engagement gain must not come at a purchase cost |
| Error rate | < 0.1% | System stability |

### Statistical Test

After 14 days, apply a **two-proportion z-test** comparing CTR in control vs treatment:

```
z = (p̂_t - p̂_c) / sqrt(p̂(1-p̂)(1/n_c + 1/n_t))
```

Reject H₀ if |z| > z_{α/2} = 1.96 (α=0.05, two-tailed). Report the 95% confidence interval
on the lift to communicate the practical significance to stakeholders.

### Decision Rule

Ship the Context-Aware model if:
1. CTR lift is statistically significant (p < 0.05), **and**
2. All guardrail metrics pass, **and**
3. The point estimate of the lift exceeds the minimum business threshold (+0.5 pp).